# Fine-tuning the retrievers (GPU)
Everything that needs a GPU, in one run: off-the-shelf embedding models on dev and test, three fine-tuned models, a variant trained with BSARD's synthetic questions, and a cross-encoder reranker. No API calls, no cost beyond Colab.

About 1 h 30 on an L4, less on an A100. At the end, `results/` and the fine-tuned model are copied to your Drive.

### Setup
The code is cloned from GitHub. Runs and models are stored in Drive, so they survive the end of the session.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/lexsearch"
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists("/content/lexsearch"):
    !git clone https://github.com/LacosteEvan/lexsearch.git /content/lexsearch
%cd /content/lexsearch
!git pull -q
!pip install -q -e .

In [ ]:
import torch
print(torch.cuda.get_device_name(0), "bf16" if torch.cuda.is_bf16_supported() else "no bf16")

### 1. Off-the-shelf models
BM25 and the first three were already run on dev on the Mac; they are run again here so that dev and test come from the same setup.

In [ ]:
BASES = "dense:intfloat/multilingual-e5-small dense:intfloat/multilingual-e5-base dense:BAAI/bge-m3 dense:Qwen/Qwen3-Embedding-0.6B"
!lexsearch retrieve bm25 $BASES --split dev
!lexsearch retrieve bm25 $BASES --split test

### 2. Fine-tuning on the 698 `fit` questions
One example per (question, answering article) with a hard negative mined from the base model, 3 epochs. Each model is then evaluated on dev (to choose) and test (for the final table).

In [ ]:
for base, out in [("intfloat/multilingual-e5-small", "e5-small-ft"),
                  ("intfloat/multilingual-e5-base", "e5-base-ft"),
                  ("BAAI/bge-m3", "bge-m3-ft")]:
    !lexsearch finetune {base} --out models/{out}
    !lexsearch retrieve dense:models/{out} --split dev
    !lexsearch retrieve dense:models/{out} --split test

### 3. Adding synthetic questions
BSARD ships 113k machine-written questions, one article each, covering the whole corpus. Do they help with articles no real question points to? 20,000 of them on top of the real ones, 1 epoch (the run is 5 times larger).

In [ ]:
!lexsearch finetune BAAI/bge-m3 --out models/bge-m3-ft-syn --synthetic 20000 --epochs 1
!lexsearch retrieve dense:models/bge-m3-ft-syn --split dev
!lexsearch retrieve dense:models/bge-m3-ft-syn --split test

### 4. Reranking and hybrid
A cross-encoder rescoring the top 50, and BM25 fused with the fine-tuned model.

In [ ]:
for split in ("dev", "test"):
    !lexsearch retrieve rerank:BAAI/bge-reranker-v2-m3@dense:BAAI/bge-m3 rerank:BAAI/bge-reranker-v2-m3@dense:models/bge-m3-ft hybrid:bm25,dense:models/bge-m3-ft --split {split}

### 5. Save to Drive
`lexsearch-results.zip` holds the rankings, the summaries and the cached article embeddings of the fine-tuned models. The models are copied as folders. Download `lexsearch-results.zip` and the folder of the best model on dev (probably `bge-m3-ft`): the search agent needs it on the Mac to embed its queries.

In [ ]:
!zip -qr {DRIVE}/lexsearch-results.zip results cache/emb/models__*
for m in os.listdir("models"):
    !rm -rf {DRIVE}/{m} && cp -r models/{m} {DRIVE}/{m}
!ls -la {DRIVE}
!python scripts/table.py dev
!python scripts/table.py test